In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGMR8B.dcf
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGMR8B.frw
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGMR8B.dat
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGIR8B.docx
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGMR8B.frq
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGMR8B/NGMR8B.map
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGKR8BDT/NGKR8BFL.dta
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGKR8BDT/NGKR8BFL.DO
/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024

# Stage 1: Exploring The Folder Structure

In [44]:
# I am exploring my folder structure (tree structure)
import os

#Going through everything in my uploaded dataset
input_path = "/kaggle/input/datasets/prospi/dhs-anc-data"

for root, dirs, files in os.walk(input_path):
    #I will skip hidden folders here
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(input_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {indent}{file}  ({size_mb:.1f} MB)")

dhs-anc-data/
  data_raw/
    nigeria_2024/
      NG_2024_DHS_02122026_1349_239942/
        NGMR8B/
          NGMR8B.dcf  (0.2 MB)
          NGMR8B.frw  (2.9 MB)
          NGMR8B.dat  (14.1 MB)
          NGIR8B.docx  (0.1 MB)
          NGMR8B.frq  (2.9 MB)
          NGMR8B.map  (0.2 MB)
        NGKR8BDT/
          NGKR8BFL.dta  (39.3 MB)
          NGKR8BFL.DO  (0.2 MB)
          NGKR8BFL.DCT  (0.1 MB)
          NGKR8BFL.frq  (5.3 MB)
          NGKR8BFL.frw  (5.3 MB)
          NGKR8BFL.map  (0.4 MB)
        NGHR8BDT/
          NGHR8BFL.dta  (166.9 MB)
          NGHR8BFL.DCT  (0.2 MB)
          NGHR8BFL.map  (0.1 MB)
          NGHR8BFL.frw  (6.6 MB)
          NGHR8BFL.DO  (0.4 MB)
          NGHR8BFL.frq  (6.6 MB)
        NGGE8AFL/
          NGGE8AFL.sbn  (0.0 MB)
          NGGE8AFL.dbf  (4.2 MB)
          NGGE8AFL.shx  (0.0 MB)
          DHS_README.txt  (0.0 MB)
          NGGE8AFL.cpg  (0.0 MB)
          NGGE8AFL.shp.xml  (0.0 MB)
          NGGE8AFL.sbx  (0.0 MB)
          NGGE8AFL.prj  

# Stage 2: Install Libraries and Preprocess

In [45]:
# Install pyreadstat
import subprocess
subprocess.run(["pip", "install", "pyreadstat", "-q"])
print ("Done")

Done


In [46]:
# Here I set the exact file paths to know where to read from
import os

BASE = "/kaggle/input/datasets/prospi/dhs-anc-data/data_raw"

NIGERIA_IR = os.path.join(
    BASE,
    "/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGIR8BDT/NGIR8BFL.dta")

KENYA_IR = os.path.join(
    BASE,
    "/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/kenya_2022/KE_2022_DHS_02122026_161_239942/KEIR8CDT/KEIR8CFL.DTA")

NIGERIA_GC = os.path.join(
    BASE,
    "/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGGC8AFL/NGGC8AFL.csv")

KENYA_GC = os.path.join(
    BASE,
    "/kaggle/input/datasets/prospi/dhs-anc-data/data_raw/kenya_2022/KE_2022_DHS_02122026_161_239942/KEGC8AFL/KEGC8AFL.csv")

# Confirm if each path resolves
for label, path in [("Nigeria IR", NIGERIA_IR), ("Kenya IR", KENYA_IR), ("Nigeria GC", NIGERIA_GC), ("Kenya GC", KENYA_GC)]:
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:1f} MB" if exists else "NOT FOUND"
    print(f"{label:12s} {exists} {size} {path}")

Nigeria IR   True 285.541335 MB /kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGIR8BDT/NGIR8BFL.dta
Kenya IR     True 214.927739 MB /kaggle/input/datasets/prospi/dhs-anc-data/data_raw/kenya_2022/KE_2022_DHS_02122026_161_239942/KEIR8CDT/KEIR8CFL.DTA
Nigeria GC   True 2.068009 MB /kaggle/input/datasets/prospi/dhs-anc-data/data_raw/nigeria_2024/NG_2024_DHS_02122026_1349_239942/NGGC8AFL/NGGC8AFL.csv
Kenya GC     True 2.553351 MB /kaggle/input/datasets/prospi/dhs-anc-data/data_raw/kenya_2022/KE_2022_DHS_02122026_161_239942/KEGC8AFL/KEGC8AFL.csv


In [47]:
# Now let's check the column names without loading the full file

import pyreadstat
import warnings
warnings.filterwarnings("ignore")

def peek_columns(filepath):
    _, meta = pyreadstat.read_dta(filepath, row_limit=1)
    return [c.lower() for c in meta.column_names]

ng_cols = peek_columns(NIGERIA_IR)
ke_cols = peek_columns(KENYA_IR)

print(f"Nigeria IR columns: {len(ng_cols)}")
print(f"Kenya IR columns:   {len(ke_cols)}")

# Check every variable the chosen methodology requires
REQUIRED = [
    "m14",    # ANC visits — the outcome
    "v012",   # age
    "v106",   # education
    "v190",   # wealth quintile
    "v025",   # urban/rural
    "v024",   # region ADM1
    "v001",   # cluster ID
    "v005",   # sample weight
    "v201",   # total children
    "v218",   # living children
    "v467b",  # barrier: permission
    "v467c",  # barrier: money
    "v467d",  # barrier: distance
    "v467f",  # barrier: alone
    "v501",   # marital status
    "v730",   # partner education (often useful)
    "v102",   # region of residence alternative
    "v113",   # source of drinking water
    "v116",   # type of toilet facility
    "v161",   # type of cooking fuel
]

print("\n{:<10} {:>10} {:>10} {}".format("Variable", "Nigeria", "Kenya", "Status"))
print("-" * 45)
for var in REQUIRED:
    in_ng = var in ng_cols
    in_ke = var in ke_cols
    if in_ng and in_ke:
        status = "OK"
    elif in_ng and not in_ke:
        status = "Nigeria missing"
    else:
        status = "BOTH missing"
    print(f"{var:<10} {str(in_ng):>10} {status}")

Nigeria IR columns: 6411
Kenya IR columns:   5925

Variable      Nigeria      Kenya Status
---------------------------------------------
m14             False BOTH missing
v012             True OK
v106             True OK
v190             True OK
v025             True OK
v024             True OK
v001             True OK
v005             True OK
v201             True OK
v218             True OK
v467b            True OK
v467c            True OK
v467d            True OK
v467f            True OK
v501             True OK
v730             True OK
v102             True OK
v113             True OK
v116             True OK
v161             True OK


In [48]:
# Checked again for the type and number of columns
print(type(ng_cols))
print(type(ke_cols))

print(len(ng_cols))
print(len(ke_cols))

<class 'list'>
<class 'list'>
6411
5925


In [49]:
# Printed out first 20 column names under Nigeria
print(ng_cols[:20])

['caseid', 'v000', 'v001', 'v002', 'v003', 'v004', 'v005', 'v006', 'v007', 'v008', 'v008a', 'v009', 'v010', 'v011', 'v012', 'v013', 'v014', 'v015', 'v016', 'v017']


In [50]:
find_anc_variable(ng_cols, "Nigeria")


[Nigeria]
Found candidates: ['m14_1', 'm14_2']
Broad matches (first 20): ['m14_1', 'm14_2', 'm14_3', 'm14_4', 'm14_5', 'm14_6', 'mm14_01', 'mm14_02', 'mm14_03', 'mm14_04', 'mm14_05', 'mm14_06', 'mm14_07', 'mm14_08', 'mm14_09', 'mm14_10', 'mm14_11', 'mm14_12', 'mm14_13', 'mm14_14']
Total broad matches: 26
M10–M19 vars (first 20): ['m10_1', 'm10_2', 'm10_3', 'm10_4', 'm10_5', 'm10_6', 'm11_1', 'm11_2', 'm11_3', 'm11_4', 'm11_5', 'm11_6', 'm13_1', 'm13_2', 'm13_3', 'm13_4', 'm13_5', 'm13_6', 'm13a_1', 'm13a_2']


In [51]:
KEEP_VARS = [
    "m14_1",
    "v012", "v106", "v190", "v025", "v024",
    "v001", "v005", "v201", "v218",
    "v467b", "v467c", "v467d", "v467f",
    "v501", "v730", "v113", "v116", "v161",
]

def load_ir(filepath, country_label, keep_vars, all_cols):
    load = [v for v in keep_vars if v in all_cols]
    skip = [v for v in keep_vars if v not in all_cols]
    if skip:
        print(f"[{country_label}] Skipping (not in file): {skip}")
    print(f"[{country_label}] Loading {len(load)} columns ...")
    df, _ = pyreadstat.read_dta(filepath, usecols=load, encoding="latin1")
    df.columns = df.columns.str.lower()
    df["country"] = country_label
    print(f"[{country_label}] Loaded {len(df):,} rows")
    return df

ng_raw = load_ir(NIGERIA_IR, "nigeria", KEEP_VARS, ng_cols)
ke_raw = load_ir(KENYA_IR,   "kenya",   KEEP_VARS, ke_cols)

[nigeria] Loading 19 columns ...
[nigeria] Loaded 39,050 rows
[kenya] Loading 19 columns ...
[kenya] Loaded 32,156 rows


In [52]:
# m14 was not found in the columns
# Knowing that the IR file has one row per woman with multiple births
# I will use m14_1 as the most recent birth instead of m14
# Update KEEP_VARS to replace m14 with m14_1

KEEP_VARS = [
    "m14_1", # ANC visits for most recent birth, replaces m14
    "v012", "v106", "v190", "v025", "v024",
    "v001", "v005", "v201", "v218",
    "v467b", "v467c", "v467d", "v467f",
    "v501", "v730", "v113", "v116", "v161",
]

# Verify that both files have it
print("m14_1 in Nigeria:", "m14_1" in ng_cols)
print("m14_1 in Kenya:", "m14_1" in ke_cols)




m14_1 in Nigeria: True
m14_1 in Kenya: True


In [53]:
def build_outcome(df, country_label, threshold=4):
    df = df.copy()
    df = df.rename(columns={"m14_1": "m14"})
    df["m14"] = pd.to_numeric(df["m14"], errors="coerce")
    df.loc[df["m14"] >= 98, "m14"] = np.nan
    before = len(df)
    df = df.dropna(subset=["m14"]).copy()
    df["anc_adequate"] = (df["m14"] >= threshold).astype(int)
    rate = df["anc_adequate"].mean() * 100
    print(f"[{country_label}] Rows: {len(df):,} | Dropped: {before-len(df):,} | ANC adequate: {rate:.1f}%")
    return df

ng = build_outcome(ng_raw, "nigeria")
ke = build_outcome(ke_raw, "kenya")

[nigeria] Rows: 13,594 | Dropped: 25,456 | ANC adequate: 53.4%
[kenya] Rows: 10,380 | Dropped: 21,776 | ANC adequate: 62.4%


In [54]:
# The number of rows dropped were almost double those that remained
# Dropped rows are women who either had no recent birth or were not asked the question
# Check the retained rows make sense demographically
# Nigeria's poorest quintile (3,648) is nearly double the richest (2,050).

for df, label in [(ng, "Nigeria"), (ke, "Kenya")]:
    print(f"\n{label}")
    print(f"  Rows retained        : {len(df):,}")
    print(f"  ANC adequate rate    : {df['anc_adequate'].mean()*100:.1f}%")
    print(f"  Age range            : {df['v012'].min():.0f} – {df['v012'].max():.0f}")
    print(f"  Wealth distribution  :\n{df['v190'].value_counts().sort_index()}")
    print(f"  Urban share          : {(df['v025']==1).mean()*100:.1f}%")


Nigeria
  Rows retained        : 13,594
  ANC adequate rate    : 53.4%
  Age range            : 15 – 49
  Wealth distribution  :
v190
1    3648
2    2815
3    2625
4    2456
5    2050
Name: count, dtype: int64
  Urban share          : 37.3%

Kenya
  Rows retained        : 10,380
  ANC adequate rate    : 62.4%
  Age range            : 15 – 49
  Wealth distribution  :
v190
1    3237
2    1761
3    1822
4    2044
5    1516
Name: count, dtype: int64
  Urban share          : 34.6%


In [55]:
# Clean DHS sentinel codes
# Protected columns are not touched
# Replace missing values with NaN
# Cleaning both datasets
# Create an urban indicator

def clean_dhs(df, country_label):
    df = df.copy() # Prevents modifying the original dataset
    protected = {"m14", "anc_adequate", "country", "v001", "v005", "v024", "v025", "v190", "v106"}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
               if c not in protected] # Clean all variables except in protected columns
    for col in num_cols:
        col_max = df[col].max()
        if col_max in [8,9]: df.loc[df[col] >= 8, col] = np.nan
        elif col_max in [98,99]: df.loc[df[col] >= 98, col] = np.nan
        elif col_max in [998, 999]: df.loc[df[col] >= 998, col] = np.nan
    df["sample_weight"] = df["v005"] / 1_000_000 # Converts weights back to real weights
    df = df.drop(columns=["v005"]) # Replace original weight with new sample_weight
    df["urban"] = (df["v025"] == 1).astype(int)
    print(f"[{country_label}] Cleaning complete")
    return df
ng = clean_dhs(ng, "nigeria")
ke = clean_dhs(ke, "kenya")

[nigeria] Cleaning complete
[kenya] Cleaning complete


In [56]:
# Finding out what would be dropped
# Preview what will be dropped before running align_and_combine
ng_only = set(ng.columns) - set(ke.columns)
ke_only = set(ke.columns) - set(ng.columns)
common  = set(ng.columns) & set(ke.columns)

print(f"Shared columns (kept)        : {len(common)}")
print(f"Nigeria-only (will be dropped): {sorted(ng_only)}")
print(f"Kenya-only (will be dropped)  : {sorted(ke_only)}")

Shared columns (kept)        : 22
Nigeria-only (will be dropped): []
Kenya-only (will be dropped)  : []


In [57]:
# Align and combine columns
# Keeps only shared columns which is essential for horizontal federated learning

def align_and_combine(ng, ke):
    common = sorted(set(ng.columns) & set(ke.columns)) # Keeping common values in each dataset for fair comparisons
    ng_only = set(ng.columns) - set(ke.columns)
    ke_only = set(ke.columns) - set(ng.columns)
    if ng_only: print(f"Nigeria-only dropped: {sorted(ng_only)}")
    if ke_only: print(f"Kenya-only dropped: {sorted(ke_only)}")
    df = pd.concat([ng[common], ke[common]], ignore_index=True)
    print(f"Combined shape : {df.shape}")
    print(f"Countries : {df['country'].value_counts().to_dict()}")
    return df

combined = align_and_combine(ng, ke)

Combined shape : (23974, 22)
Countries : {'nigeria': 13594, 'kenya': 10380}


In [58]:
# Handling missing values
# Data preprocessing pipeline


from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

DROP_THRESH  = 0.30 # Drop column if missing above 30%
MEDIAN_THRESH = 0.05 # 0-5% fill with median

def handle_missing(df):
    df = df.copy()
    meta_cols = {"anc_adequate", "m14", "country", "v001", "sample_weight",
                 "v024", "v025", "urban", "v190", "v106"}
    feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                 if c not in meta_cols] # Only numberical values are cleaned
    rates = df[feat_cols].isnull().mean().sort_values(ascending=False) # Calculates % of missing values per column

    # Report columns that have missing values
    has_missing = rates[rates > 0]
    print("Missingness:" if len(has_missing) else "No missing values.")
    if len(has_missing):
        print(has_missing.mul(100).round(1).to_string())

    # Tier 0: drop >30%
    drop_cols = rates[rates > DROP_THRESH].index.tolist()
    if drop_cols:
        print(f"\nDropping (>30% missing): {drop_cols}")
        df = df.drop(columns=drop_cols)
        rates = rates.drop(drop_cols)
        feat_cols = [c for c in feat_cols if c not in drop_cols]

    # Tier 1: median impute <5%
    low_cols = rates[rates <= MEDIAN_THRESH].index.tolist()
    if low_cols:
        df[low_cols] = df[low_cols].fillna(df[low_cols].median())
        print(f"Median-imputed: {len(low_cols)} columns")

    # Tier 2: iterative impute 5–30%
    mid_cols = rates[(rates > MEDIAN_THRESH) & (rates <= DROP_THRESH)].index.tolist()
    if mid_cols:
        print(f"Iterative-imputing: {mid_cols}")
        imp = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, random_state=42),
            max_iter=5, random_state=42
        )
        df[mid_cols] = imp.fit_transform(df[mid_cols]) # Replaces missing values with model predictions
        print("Done")
    return df

combined = handle_missing(combined) # Applies the cleaning to the merged dataset

No missing values.
Median-imputed: 7 columns


In [59]:
# Verify previous changes and save

import os

print("=" * 45)
print(f"Shape            : {combined.shape}") # Shows dataset size

print("\nANC rate by country:") # ANC average by country
print(combined.groupby("country")["anc_adequate"]
      .agg(N="count", ANC_pct=lambda x: round(x.mean()*100, 1)))

print("\nANC rate by wealth quintile:") # Inequality in healthcare access
print(combined.groupby("v190")["anc_adequate"].mean().mul(100).round(1))

print("\nANC rate by urban/rural (1=urban 0=rural):") # ANC by urban vs rural
print(combined.groupby("urban")["anc_adequate"].mean().mul(100).round(1))

print("\nRemaining missing:") # Keeps columns that still have missing values if any
mv = combined.isnull().sum()
mv = mv[mv > 0]
print(mv if len(mv) else "None")

os.makedirs("/kaggle/working/processed", exist_ok=True) # Creates folder to save file as parquet
save_path = "/kaggle/working/processed/combined_clean.parquet"
combined.to_parquet(save_path, index=False) 
print(f"\nSaved : {save_path}")
print(f"Size  : {os.path.getsize(save_path)/1e6:.1f} MB") # Confirms the saved file

Shape            : (23974, 22)

ANC rate by country:
             N  ANC_pct
country                
kenya    10380     62.4
nigeria  13594     53.4

ANC rate by wealth quintile:
v190
1    37.3
2    47.6
3    60.0
4    72.3
5    86.2
Name: anc_adequate, dtype: float64

ANC rate by urban/rural (1=urban 0=rural):
urban
0    49.0
1    71.9
Name: anc_adequate, dtype: float64

Remaining missing:
v467b    4941
v467c    4941
v467d    4941
v467f    4941
v730     2669
dtype: int64

Saved : /kaggle/working/processed/combined_clean.parquet
Size  : 0.2 MB


In [60]:
# There were some 4,941 missing values in barrier variables and 2,669 in v730
# These values appeared to have low missingness but were actually still dirty

# Force clean remaining missing values
barrier_cols = ["v467b", "v467c", "v467d", "v467f"]
partner_col  = ["v730"]

# Barrier variables: missing means the question was not asked
# (woman had no barrier or was not in the reference group)
# Treat as 0 = no barrier reported
combined[barrier_cols] = combined[barrier_cols].fillna(0)

# Partner education: fill with median
combined["v730"] = combined["v730"].fillna(combined["v730"].median())

# Confirm
mv = combined.isnull().sum()
mv = mv[mv > 0]
print("Remaining missing after fix:")
print(mv if len(mv) else "None — all clean")
print(f"\nShape: {combined.shape}")

Remaining missing after fix:
None — all clean

Shape: (23974, 22)


In [61]:
# Now to resave the file

combined.to_parquet("/kaggle/working/processed/combined_clean.parquet", index=False)
print("Saved cleanly")

Saved cleanly
